In [ ]:
from mp_api.client import MPRester
import numpy as np
import pickle
import pandas as pd
from tqdm import tqdm
from pymatgen.analysis.diffraction.xrd import XRDCalculator


def XRD_gaussian(xrd_pattern,sig):
    x = np.linspace(0,90,901)
    y = np.zeros(901)
    for i in range(len(xrd_pattern)):
        y = y + xrd_pattern.y[i] * gaussian(x, xrd_pattern.x[i], sig)
    return y

def gaussian(x, mu, sig):
    return 1./(np.sqrt(2.*np.pi)*sig)*np.exp(-np.power((x - mu)/sig, 2.)/2)

In [ ]:
xrd_calculator = XRDCalculator()
mpr=MPRester("id")
import os
dic=[]
for i in list(range(230))[::-1]:
    data = mpr.materials.summary.search(spacegroup_number=i+1, fields=['material_id','structure',"elements"])

    for one in tqdm(data):
        structure = one.structure
        xrd_pattern = xrd_calculator.get_pattern(structure)
        XRDgaussian = XRD_gaussian(xrd_pattern,0.05)
        temp={"material_id":one.material_id,'XRDgaussian':XRDgaussian,"elements":one.elements}
        dic.append(temp)
        # os.makedirs('/home/light/PycharmProjects/material/data/crystallization'+str(i+1),exist_ok=True)
        # temp.to_csv('/home/light/PycharmProjects/material/data/crystallization'+str(i+1)+'/xrd_'+str(one.material_id)+'.csv',index=False)
    os.makedirs('/filename/data_sg',exist_ok=True)
    pickle.dump(dic,open(f'/filename/data_sg/spacegroup{i+1}.pkl',"wb"))
